In [ ]:
import os
%pwd
os.chdir("../")
%pwd

'c:\\Users\\Hp\\Code\\mlops-chest-cancer-classification'

In [16]:
%ls


 Volume in drive C has no label.
 Volume Serial Number is 4230-F99E

 Directory of c:\Users\Hp\Code\mlops-chest-cancer-classification

01/31/2026  10:56 AM    <DIR>          .
01/23/2026  09:21 PM    <DIR>          ..
01/25/2026  03:35 PM               151 .env
01/29/2026  03:13 PM    <DIR>          .github
01/29/2026  04:17 PM             4,895 .gitignore
01/29/2026  02:55 PM    <DIR>          .ipynb_checkpoints
01/31/2026  10:42 AM    <DIR>          artifacts
01/29/2026  03:13 PM    <DIR>          config
01/29/2026  03:13 PM                 0 dvc.yaml
01/23/2026  11:35 PM             1,092 LICENSE
01/29/2026  04:24 PM    <DIR>          logs
01/31/2026  10:48 AM               479 main.py
01/31/2026  02:07 PM               159 params.yaml
01/30/2026  09:33 PM               383 README.md
01/29/2026  04:10 PM               186 requirements.txt
01/31/2026  10:58 AM    <DIR>          research
01/29/2026  03:48 PM               834 setup.py
01/29/2026  04:18 PM    <DIR>          src
01/29/2

In [18]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path 
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int


In [19]:
from chestCancerClassifier.constants import *
from chestCancerClassifier.utils.common import read_yaml, create_directories

In [39]:
class ConfigurationManager:
    def __init__(
        self,
        config_path = CONFIG_FILE_PATH,
        params_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])

    def prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model

        create_directories([config.root_dir])
        
        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_classes=self.params.CLASSES,
            params_weights=self.params.WEIGHTS,
            params_include_top=self.params.INCLUDE_TOP)
        
        return prepare_base_model_config

In [40]:
import os
from zipfile import ZipFile
import urllib.request as request
from chestCancerClassifier import logger
import tensorflow as tf

In [41]:
class PrepareBaseModel:
    def __init__(self, config:PrepareBaseModelConfig):
        self.config = config
    
    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape = self.config.params_input_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(path=self.config.base_model_path, model=self.model)

    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            for layer in model.layers:
                model.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                model.trainable = False

        flattern_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.model.Model(
            inputs=model.input,
            outputs=predictions
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )
         
        full_model.summary()
        return full_model

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classe=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=elf.config.params_learning_rate
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)

    @staticmethod
    def save_model(path: Path, model:tf.keras.Model):
        model.save(path)

In [38]:
 try:
    config = ConfigurationManager()
    prepare_base_model_config = config.prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e
    

[2026-01-31 16:03:40,940: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-01-31 16:03:40,943: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-31 16:03:40,945: INFO: common: created directory at artifacts]
[2026-01-31 16:03:40,945: INFO: common: created directory at artifacts/prepare_base_model]


TypeError: PrepareBaseModelConfig.__init__() got an unexpected keyword argument 'upadated_base_model_path'